# Анализ оттока клиентов телеком-оператора

Оператор теряет часть клиентов каждый месяц, и его интересует не сам факт оттока, а **кто уходит и почему** — чтобы понять, кого удерживать и чем.

В ноутбуке я прохожу полный цикл: готовлю данные, смотрю на них, проверяю пару гипотез, строю простую модель и в конце формулирую рекомендации.

Данные синтетические и генерируются ниже с фиксированным сидом, поэтому ноутбук запускается без внешних файлов и даёт тот же результат. По структуре они повторяют известный датасет *Telco Customer Churn* с Kaggle.


## 1. Данные


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 2000

contract = rng.choice(['Month-to-month', 'One year', 'Two year'], size=n, p=[0.55, 0.25, 0.20])
internet = rng.choice(['DSL', 'Fiber optic', 'No'], size=n, p=[0.35, 0.45, 0.20])
payment  = rng.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'],
                      size=n, p=[0.35, 0.22, 0.22, 0.21])
tenure   = rng.integers(1, 72, size=n)                      # месяцев с оператором
monthly  = np.round(rng.normal(65, 30, size=n).clip(20, 120), 2)
total    = np.round(tenure * monthly * rng.normal(1.0, 0.05, size=n), 2)

# Вероятность оттока зависит от типа контракта, срока и суммы счёта
contract_effect = pd.Series(contract).map(
    {'Month-to-month': 1.5, 'One year': 0.2, 'Two year': -1.0}).to_numpy()
logit = (-2.0 + contract_effect - 0.03 * tenure + 0.01 * (monthly - 65)
         + rng.normal(0, 0.4, size=n))
prob  = 1 / (1 + np.exp(-logit))
churn = (rng.random(n) < prob).astype(int)

df = pd.DataFrame({
    'customerID': [f'C{i:05d}' for i in range(n)],
    'tenure': tenure, 'Contract': contract, 'InternetService': internet,
    'PaymentMethod': payment, 'MonthlyCharges': monthly, 'TotalCharges': total,
    'Churn': churn,
})
df.head()


## 2. Очистка и проверка

Смотрю на типы, пропуски и очевидные аномалии, прежде чем что-то считать.


In [ ]:
print('Размер:', df.shape)
print('Пропуски:')
print(df.isna().sum())
print('\nДоля оттока:', round(df['Churn'].mean(), 3))
df.describe(numeric_only=True)


## 3. Разведочный анализ

Главный вопрос — от чего зависит отток. Начинаю с типа контракта и срока жизни клиента.


In [ ]:
# Отток по типу контракта
by_contract = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
print('Доля оттока по контракту:')
print(by_contract.round(3))


In [ ]:
# Отток по сроку жизни клиента (бакеты tenure)
df['tenure_bucket'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72],
                             labels=['0-12', '12-24', '24-48', '48-72'])
by_tenure = df.groupby('tenure_bucket', observed=True)['Churn'].mean()
print('Доля оттока по сроку жизни:')
print(by_tenure.round(3))


Что видно на данных: отток сильно выше у клиентов на помесячном контракте и у новичков (первый год). Это две главные группы риска, и дальше я это проверю статистикой.


## 4. Проверка гипотез

**Гипотеза 1:** тип контракта связан с оттоком. Проверяю хи-квадратом.

**Гипотеза 2:** у ушедших выше месячный счёт. Проверяю t-тестом.


In [ ]:
from scipy import stats

# H1: контракт и отток независимы?
ct = pd.crosstab(df['Contract'], df['Churn'])
chi2, p1, dof, _ = stats.chi2_contingency(ct)
print(f'Хи-квадрат: chi2={chi2:.1f}, p={p1:.2e}')

# H2: месячный счёт у ушедших и оставшихся
a = df.loc[df['Churn'] == 1, 'MonthlyCharges']
b = df.loc[df['Churn'] == 0, 'MonthlyCharges']
t, p2 = stats.ttest_ind(a, b, equal_var=False)
print(f'Средний счёт: ушедшие={a.mean():.1f}, оставшиеся={b.mean():.1f}')
print(f't-тест: t={t:.1f}, p={p2:.2e}')


При p намного меньше 0.05 обе связи значимы: тип контракта связан с оттоком, и у ушедших счёт в среднем выше.


## 5. Модель

Строю логистическую регрессию — она не только предсказывает отток, но и показывает, какие признаки на него влияют и в какую сторону.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Contract',
            'InternetService', 'PaymentMethod']
X = pd.get_dummies(df[features], drop_first=True)
y = df['Churn']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)

proba = model.predict_proba(X_te)[:, 1]
print('ROC AUC:', round(roc_auc_score(y_te, proba), 3))
print(classification_report(y_te, model.predict(X_te)))


In [ ]:
# Влияние признаков: коэффициенты модели
coef = pd.Series(model.coef_[0], index=X.columns).sort_values()
print('Снижают отток (<0) / повышают (>0):')
print(coef.round(2))


## 6. Выводы и рекомендации

Что показывают данные и модель:

- **Помесячный контракт — главный фактор риска.** Перевод таких клиентов на годовой контракт (скидка за переход) напрямую снижает отток.
- **Первый год — зона риска.** Новичкам нужен отдельный онбординг и внимание в первые месяцы, пока они не закрепились.
- **Высокий месячный счёт повышает отток.** Таким клиентам стоит показывать ценность тарифа или предлагать более подходящий пакет, пока они не ушли.

Что с этим делать дальше: модель даёт вероятность оттока по каждому клиенту, её можно выгружать в CRM и запускать удержание по списку с самым высоким риском, а не по всем сразу.
